# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the dataset [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This helps you understand the organization and contents of the dataset.

In [ ]:
# List all record sets and their fields by @id
print("Available record sets:")
for recset in metadata.record_sets:
    print(f"- RecordSet name: {recset.name}\n  @id: {recset.id}")
    print("  Fields:")
    for field in recset.fields:
        print(f"    - Field name: {field.name}   @id: {field.id}   Data type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use record set and field `@id`s from the overview above.

In [ ]:
# List all record set @ids
record_sets = [recset.id for recset in metadata.record_sets]
print(f"Record Set @ids: {record_sets}")

dataframes = {}

for record_set_id in record_sets:
    # Load all records of the record set
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from RecordSet {record_set_id}")

# Example: Display the columns and head for the main data record set
# We'll pick the largest (likely main) set by number of columns
max_cols = 0
main_recset_id = None
for recid, df in dataframes.items():
    if df.shape[1] > max_cols:
        main_recset_id = recid
        max_cols = df.shape[1]
if main_recset_id:
    print(f"\nColumns in main RecordSet ({main_recset_id}):")
    print(dataframes[main_recset_id].columns.tolist())
    display(dataframes[main_recset_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming distributions, or aggregating by key attributes.

In [ ]:
# Pick a numeric field to analyze (e.g., 'cr:field:mw_kda' for molecular weight in kiloDaltons)
# List all numeric candidate fields for demonstration
numeric_candidates = []
fields = []
for rec in metadata.record_sets:
    if rec.id == main_recset_id:
        fields = rec.fields
        for field in rec.fields:
            if field.data_type in ['Float', 'Number', 'Integer']:
                numeric_candidates.append((field.name, field.id))
print('Numeric fields in main record set:')
for name, fid in numeric_candidates:
    print(f"- {name}: {fid}")

# For demonstration, let's pick the first numeric field
if numeric_candidates:
    numeric_field_id = numeric_candidates[0][1]
else:
    raise Exception('No numeric field found!')
print(f"Using numeric field: {numeric_field_id}")

# Filtering records where numeric_field > threshold
threshold = 10
df = dataframes[main_recset_id]
# Coerce to numeric if necessary
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
display(filtered_df.head())

# Normalize field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt grouping by a key attribute (e.g., sample type or similar, if available)
# Let's look for a categorical field
cat_candidates = []
for field in fields:
    if field.data_type in ['Text', 'Boolean'] and field.id != numeric_field_id:
        cat_candidates.append((field.name, field.id))
if cat_candidates:
    group_field_id = cat_candidates[0][1]
    print(f"Grouping by field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped by {group_field_id} (showing mean of {numeric_field_id}):")
    display(grouped_df.head())
else:
    print("No suitable categorical field for grouping found.")

## 5. Visualization
Visualize distributions of numeric data or relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field after filtering
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.show()

# If grouped data available, barplot the means
if 'grouped_df' in locals():
    plt.figure(figsize=(10,5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the [FAIR² Croissant](https://sen.science/doi/10.71728/senscience.vq4a-28xa) schema, explored its main record set and fields, extracted records, filtered by a key numeric feature, normalized it, grouped by a categorical attribute (if available) and visualized the resulting distributions.

**Key findings:**
- The dataset provides rich protein feature data from mass spectrometry analysis, accessible by `@id` references.
- The `mlcroissant` library provides a flexible way to dynamically explore record sets, fields, and extract subsets for EDA.
- Further analysis can include downstream protein annotation, biomarker discovery, or experiment tracking using grouped/categorized fields.

Refer to the Croissant documentation and dataset [schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) for full details on field meaning, types, and update cycles.